# Custom prediction (not x7 / x20)

Misma lógica que `run_prediction`, pero con **`CUSTOM_MODELS`** en `curve_config.py`.

Ejemplo ya configurado: `2s10s ~ SFRH7`

### Cómo añadir una serie que no está en la DB
1. Estructura swap → `CUSTOM_CURVES` / `CUSTOM_FLIES`  (p.ej. `("2s10s", 2, 10)` → BBG `USOS0210 Curncy`)
2. Otro ticker BBG → `CROSS_ASSET`  (p.ej. `("SFRH7", "SFRH7 Comdty")`)
3. Modelo → `CUSTOM_MODELS`
4. `git pull` + celda de abajo con `PULL_HISTORY = True` una vez

### Correlaciones
La lista de la matriz está en **`CORR_SERIES`** (mismo `curve_config.py`). Añade ahí los nombres limpios que quieras (`2s10s`, `SFRH7`, `5s10s`, …).

In [ ]:
import importlib
import logging
import os
import sys
from pathlib import Path

from IPython.display import HTML, display
import pandas as pd

logging.disable(logging.WARNING)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

_DZ = Path("/v/region/na/appl/datazone/notebooks2/data/users/arraipau/fwd_screener_v1")
REPO = Path.cwd() if (Path.cwd() / "run_custom_prediction.py").exists() else _DZ
if not (REPO / "run_custom_prediction.py").exists():
    raise FileNotFoundError("No encuentro run_custom_prediction.py — git pull")

os.chdir(REPO)
sys.path = [str(REPO)] + [p for p in sys.path if str(Path(p).resolve()) != str(REPO.resolve())]
print("REPO:", REPO.resolve())

In [ ]:
import run_dz

importlib.reload(run_dz)
run_dz.bootstrap_for_repo(str(REPO))

In [ ]:
REFRESH = True
PULL_HISTORY = True   # True la primera vez (o al añadir tickers nuevos)
MAKE_GRAPHS = True

import curve_config
import run_custom_prediction

importlib.reload(curve_config)
importlib.reload(run_custom_prediction)

print("CUSTOM_CURVES:", curve_config.CUSTOM_CURVES)
print("CUSTOM_FLIES :", curve_config.CUSTOM_FLIES)
print("CROSS_ASSET :", curve_config.CROSS_ASSET)
print("CUSTOM_MODELS:")
for m in curve_config.CUSTOM_MODELS:
    print(" ", m.name, ":", m.target, "~", m.features)

In [ ]:
table = run_custom_prediction.run_once(
    refresh=REFRESH,
    force_history=PULL_HISTORY,
    make_graphs=MAKE_GRAPHS,
)
from models import PREDICTION_COLS
cols = [c for c in PREDICTION_COLS if c in table.columns]
display(table[cols])

In [ ]:
html = run_custom_prediction.preview_html()
print("asof:", run_custom_prediction.LAST_ASOF, "graphs:", len(run_custom_prediction.LAST_GRAPH_PATHS))
display(HTML(html))